
# 練習問題: schedule(dynamic) で負荷を均す

## 目標

すでに並列化されたループに `schedule(dynamic)` 句を1つ追加するだけで, 繰り返しごとの仕事量が大きく異なる (アンバランスな) ループの負荷をスレッド間で均等に分担できることを体験する.

## 課題

`schedule_dynamic.cpp` (または `schedule_dynamic.f90`) のループは, 各繰り返し `i` の仕事量が `i` に比例して増える (内側ループが `i` 回まわる) ため, デフォルトの分割 (static) では, 大きい `i` を割り当てられたスレッドだけが長く働き, 他のスレッドが待たされる.

コメント `TODO` の箇所で **ループを並列化し, `schedule(dynamic)` で負荷分散せよ**.

- C++: `#pragma omp parallel for schedule(dynamic) reduction(+:total)`
- Fortran: `!$omp parallel do schedule(dynamic) reduction(+:total)` … `!$omp end parallel do`

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore schedule_dynamic.cpp -o schedule_dynamic.exe

# Fortran
nvfortran -fast -mp=multicore schedule_dynamic.f90 -o schedule_dynamic.exe
```

`schedule(dynamic)` を入れる前と後で, 実行時間を `time` で比較せよ.

```
OMP_NUM_THREADS=4 time ./schedule_dynamic.exe
```

## 期待される結果

- 計算結果 `total` は schedule の指定に関わらず同じ.
- デフォルト (static) では一部のスレッドに重い繰り返しが偏り, 実行時間が長くなる.
- `schedule(dynamic)` を追加すると, 空いたスレッドが順に繰り返しを取りに行くので負荷が均され, 実行時間が短くなる.

`OMP_NUM_THREADS` の値を変えて, 短縮の効果を確認せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ schedule_dynamic.cpp
#include <cstdio>
#include <omp.h>

/* 仕事量が引数 k に比例するダミー計算 (k に比例した回数だけ加算する) */
double work(long k) {
  double s = 0.0;
  for (long j = 0; j < k; j++) {
    s += 1.0 / (1.0 + j);
  }
  return s;
}

int main() {
  const int n = 2000;
  double total = 0.0;
  // TODO: 下のループを並列化し, 仕事量が i に比例して不均一なので schedule(dynamic) で負荷を均せ.
  for (int i = 0; i < n; i++) {
    /* 繰り返し i の仕事量は i に比例して重くなる (アンバランス) */
    total += work((long)i * 100000L);
  }
  printf("total = %f\n", total);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore schedule_dynamic.cpp -o schedule_dynamic_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./schedule_dynamic_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ schedule_dynamic.f90
module work_mod
contains
  ! 仕事量が引数 k に比例するダミー計算 (k に比例した回数だけ加算する)
  function work(k) result(s)
    integer(8), intent(in) :: k
    real(8) :: s
    integer(8) :: j
    s = 0.0d0
    do j = 0, k - 1
       s = s + 1.0d0 / (1.0d0 + j)
    end do
  end function work
end module work_mod

program schedule_dynamic
  use omp_lib
  use work_mod
  integer, parameter :: n = 2000
  integer :: i
  real(8) :: total
  total = 0.0d0
  ! TODO: 下のループを並列化し, 仕事量が i に比例して不均一なので schedule(dynamic) で負荷を均せ.
  do i = 0, n - 1
     ! 繰り返し i の仕事量は i に比例して重くなる (アンバランス)
     total = total + work(int(i, 8) * 100000_8)
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  print "(a,f0.6)", "total = ", total
end program schedule_dynamic

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore schedule_dynamic.f90 -o schedule_dynamic_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./schedule_dynamic_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:schedule_dynamic.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:schedule_dynamic.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:schedule_dynamic.cpp}